# RadHarmony — Evaluation Exercises: Solutions

Answer key for the **try it yourself** cells and the **build your own** template in
[`radharmony_hands_on_evaluation.ipynb`](radharmony_hands_on_evaluation.ipynb). Each cell
below is that exercise with every blank (`...`, `make_YOUR_MODEL`, …) filled in and a note
on *why*. The cells are self-contained — run **Setup** first, then any answer.

## Setup

In [ ]:
import os

# Pick the GPU BEFORE torch/CUDA is imported (device_map="auto" in the VLM answers
# should see only this device).
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
DEVICE = "cuda"  # after the mask, the chosen GPU is cuda:0

### A — a different encoder, a different dataset

RAD-DINO linear-probed on VinDr-CXR (28-label, multi-label). A classification probe needs one vector per image, so `output_keys={"img", "cls"}`, and the dataset must emit labels (`output_cls=True`).

In [ ]:
from radharmony.evaluator import LinearProbeEvaluator
from radharmony.evaluator.backbones import make_raddino
from radharmony.dataset import VinDrCXRTestDataset

# Classification probe -> the encoder emits ONE vector per image: output_keys={"img", "cls"}.
transform, encoder = make_raddino(device=DEVICE, output_keys={"img", "cls"})

VINDR_DIR = "/mnt/NAS4/datasets/external/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/test/"
vindr_ds = VinDrCXRTestDataset(
    base_image_dir=VINDR_DIR,
    transform=transform,
    output_cls=True,   # a probe reads the class labels
    cache_dir=None,
)

probe = LinearProbeEvaluator(
    image_encoder=encoder,
    dataset=vindr_ds,
    n_folds=5,
    batch_size=32,
    device=DEVICE,
    embedding_cache="./cache/vindr_raddino_emb",
)
results = probe.evaluate()
(
    results.groupby("label")[["auroc", "auprc", "f1"]]
    .mean()
    .sort_values("auroc", ascending=False)
    .head(10)
)

### B — zero-shot, no probe training

BiomedCLIP scoring RSNA Pneumonia by text prompts. `make_biomed_clip` returns `(transform, image_encoder, text_encoder, tokenizer)`; `ZeroShotEvaluator` needs the 3rd and 4th. `labels=` is restricted to the two classes we wrote prompts for.

In [ ]:
from radharmony.evaluator import ZeroShotEvaluator
from radharmony.evaluator.backbones import make_biomed_clip
from radharmony.dataset import RSNAPneumoniaKaggleTrainDataset

# make_biomed_clip -> (transform, image_encoder, text_encoder, tokenizer).
transform, encoder, text_encoder, tokenizer = make_biomed_clip(device=DEVICE)

RSNA_DIR = "/mnt/NAS4/datasets/external/rsna-pneumonia-detection-challenge/stage_2_train_images/"
RSNA_CSV = "/mnt/NAS4/datasets/external/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
zs_ds = RSNAPneumoniaKaggleTrainDataset(
    base_image_dir=RSNA_DIR,
    csv_path=RSNA_CSV,
    transform=transform,
    output_cls=True,
    cache_dir=None,
)

ev = ZeroShotEvaluator(
    image_encoder=encoder,
    text_encoder=text_encoder,   # 3rd object from the recipe
    tokenizer=tokenizer,         # 4th object from the recipe
    dataset=zs_ds,
    labels=["lung_opacity", "normal"],   # two of RSNA's three LABEL_COLS
    prompts={
        "lung_opacity": ["a chest x-ray showing lung opacity", "chest x-ray with pneumonia"],
        "normal": ["a normal chest x-ray", "clear lungs, no acute abnormality"],
    },
    negative_prompts={
        "lung_opacity": ["a normal chest x-ray with clear lungs"],
    },
    embedding_cache="./cache/rsna_biomedclip_emb",
)
ev.evaluate()

### C — segment a different finding with a different encoder

RAD-DINO (segmentation mode, `output_keys={"img", "mask"}`) probed on SIIM-ACR Pneumothorax. `num_classes=2` (background + pneumothorax).

In [ ]:
from radharmony.evaluator import LinearProbeSegEvaluator
from radharmony.evaluator.backbones import make_raddino
from radharmony.dataset import SIIMACRPTXTrainDataset

# Segmentation -> dense feature maps: output_keys must include "mask".
seg_transform, seg_encoder = make_raddino(device=DEVICE, output_keys={"img", "mask"})

SIIM_DIR = "/mnt/NAS3/datasets/external/SIIM_ACR_Pneumothorax/dicom-images-train/"
SIIM_CSV = "/mnt/NAS3/datasets/external/SIIM_ACR_Pneumothorax/train-rle.csv"
siim_ds = SIIMACRPTXTrainDataset(
    base_image_dir=SIIM_DIR,
    csv_path=SIIM_CSV,
    transform=seg_transform,
    output_mask=True,
    mask_output_dir="./cache/siim_ptx_masks",
    cache_dir=None,
)

seg_probe = LinearProbeSegEvaluator(
    image_encoder=seg_encoder,
    dataset=siim_ds,
    num_classes=2,   # background + pneumothorax
    n_folds=5,
    epochs=20,
    batch_size=8,
    device=DEVICE,
)
seg_probe.evaluate().groupby("label")[["dice", "iou"]].mean()

### D — a different VQA model

CheXagent-2 through the same `VQAEvaluator`. Its first positional argument is the `answerer` callable the recipe returns.

In [ ]:
from radharmony.evaluator import VQAEvaluator
from radharmony.evaluator.backbones import make_chexagent_vqa
from radharmony.dataset import VQARadDataset

VQA_RAD_DIR = "/mnt/NAS4/datasets/external/VQA-RAD/VQA_RAD Image Folder"

transform, answerer = make_chexagent_vqa(device=DEVICE)
vqa_ds = VQARadDataset(base_image_dir=VQA_RAD_DIR, transform=transform, cache_dir=None)

# VQAEvaluator's first positional argument is the answerer callable.
ev = VQAEvaluator(answerer, dataset=vqa_ds, output_dir="outputs/vqa_chexagent")
ev.evaluate()

### E — a different report generator

CheXagent-2 through `ReportGenerationEvaluator`. `ref_section="findings"` scores the FINDINGS section; `metrics=["bleu", "radgraph"]` pairs n-gram overlap with RadGraph F1.

In [ ]:
from radharmony.evaluator import ReportGenerationEvaluator
from radharmony.evaluator.backbones import make_chexagent_generator
from radharmony.dataset import MIMICCXRDataset

MIMIC_IMAGE_DIR = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/files/"
MIMIC_REPORT_CSV = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/cxr-study-list.csv.gz"

transform, generator = make_chexagent_generator(device=DEVICE)
gen_ds = MIMICCXRDataset(
    base_image_dir=MIMIC_IMAGE_DIR,
    report_csv_path=MIMIC_REPORT_CSV,
    transform=transform,
    output_report=True,
    cache_dir=None,
)

ev = ReportGenerationEvaluator(
    generator,
    dataset=gen_ds,
    ref_section="findings",           # score the FINDINGS section
    metrics=["bleu", "radgraph"],     # BLEU (n-gram) + RadGraph F1
    max_samples=1000,
    output_dir="outputs/chexagent_report",
)
ev.evaluate()

## Build your own — worked example

The template solved for a fresh model+dataset pairing: **DINOv3** (a general-purpose ViT, vision-only → 2-tuple) linear-probed on **Montgomery**. Retarget it by swapping the three imports and the paths.

In [ ]:
# A fully worked instance of the template: DINOv3 (a general-purpose ViT) linear-probed
# on Montgomery TB. Swap the three imports + paths to retarget any model/dataset/task.
from radharmony.evaluator import LinearProbeEvaluator
from radharmony.evaluator.backbones import make_dinov3
from radharmony.dataset import MontgomeryCXRDataset

# make_dinov3 is vision-only -> (transform, encoder).
transform, encoder = make_dinov3(device=DEVICE, output_keys={"img", "cls"})

ds = MontgomeryCXRDataset(
    base_image_dir="/mnt/NAS4/datasets/external/Montgomery-CXR/MontgomerySet/CXR_png",
    transform=transform,
    output_cls=True,
    cache_dir=None,
)

ev = LinearProbeEvaluator(image_encoder=encoder, dataset=ds, n_folds=5, device=DEVICE)
ev.evaluate().groupby("label")[["auroc", "auprc", "f1"]].mean()